# ChatPromptTemplate과 MessagesPlaceholder

메시지 틀과 선택적 대화 기록을 사용하면 같은 프롬프트 구조를 어떻게 재사용할 수 있을까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 메시지 템플릿 만들기

`MessagesPlaceholder(optional=True)`는 기록이 없는 첫 요청도 같은 틀로 처리합니다.

In [5]:
# optional placeholder는 기록이 없는 첫 요청에서도 메시지 틀을 완성합니다.
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "고객 지원 답변을 {format_rule} 형식으로 작성하세요."),
    MessagesPlaceholder("history", optional=True),
    ("human", "{question}"),
])
base_values = {"format_rule": "분류/다음 행동/고객 문장", "question": TASK}
first_messages = prompt.invoke(base_values).to_messages()
print([(m.type, m.text[:80]) for m in first_messages])

[('system', '고객 지원 답변을 분류/다음 행동/고객 문장 형식으로 작성하세요.'), ('human', '고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.\n새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 ')]


## 기록 없음과 한 건 있음 비교

과거 고객 요청 한 건만 추가하고 현재 질문과 평가 기준은 유지합니다.

In [6]:
# 기록 없음과 한 턴 기록의 완성 메시지를 각각 호출합니다.
first = run(first_messages)
with_history = prompt.invoke({**base_values, "history": [
    ("human", "답변을 세 줄 이내로 써 주세요."),
    ("ai", "알겠습니다. 이후 답변을 세 줄 이내로 작성하겠습니다."),
]}).to_messages()
second = run(with_history)
print("메시지 수:", len(first_messages), len(with_history))

메시지 수: 2 4


## 실행 결과 확인

템플릿에 채워진 메시지와 모델 응답을 구분하여 읽습니다.

In [7]:
# 기록 없음과 한 턴 기록의 응답 원문과 사용량을 메시지 수와 함께 비교합니다.
for name, result in {"history_empty": first, "history_one_turn": second}.items():
    print("\n", name, mention_flags(result["text"]), result["usage"], result["latency_s"])
    print(result["text"])


 history_empty {'교환': True, 'A-1042': True, '사진': True} {'input_tokens': 108, 'output_tokens': 96, 'total_tokens': 204, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.201
- **분류:** 파손/교환 요청
- **필요한 다음 행동:** 파손된 뚜껑 사진을 첨부해 주세요. 사진 확인 후 교환 접수를 진행하겠습니다.
- **고객에게 보낼 한 문장:** 주문번호 A-1042 확인했습니다. 깨진 뚜껑 사진을 보내주시면 새 상품 교환 절차를 바로 도와드리겠습니다.

 history_one_turn {'교환': True, 'A-1042': True, '사진': True} {'input_tokens': 146, 'output_tokens': 100, 'total_tokens': 246, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}} 1.462
분류: 교환/파손 접수  
필요한 다음 행동: 파손된 뚜껑 사진을 첨부해 주시면 교환 접수를 진행하겠습니다.  
고객에게 보낼 한 문장: 주문번호 A-1042의 텀블러 뚜껑 파손으로 교환을 원하시면, 깨진 부분이 확인되는 사진을 보내주시면 바로 접수 도와드리겠습니다.
